In [1]:
import json

def convert_weflow_to_instruction(raw_data):
    messages = raw_data.get("messages", [])
    formatted_data = []
    
    # 假设你微调的目标是模拟 senderUsername 为 "wxid_evomiks33xqs22" 的人 (isSend: 1)
    # instruction: 对方说的话
    # output: 你（isSend: 1）回的话
    
    i = 0
    while i < len(messages):
        # 1. 寻找“对方”发起的对话作为 instruction
        if messages[i]["isSend"] == 0:
            instruction_content = []
            # 合并对方连续发送的消息
            while i < len(messages) and messages[i]["isSend"] == 0:
                content = messages[i].get("content")
                if content and messages[i]["type"] == "文本消息":
                    instruction_content.append(content)
                i += 1
            
            # 2. 寻找紧接着的“你”的回复作为 output
            output_content = []
            while i < len(messages) and messages[i]["isSend"] == 1:
                content = messages[i].get("content")
                if content and messages[i]["type"] == "文本消息":
                    output_content.append(content)
                i += 1
            
            # 3. 只有当双方都有文本对话时，才记录为一条样本
            if instruction_content and output_content:
                formatted_data.append({
                    "instruction": "\n".join(instruction_content),
                    "input": "", # 如果没有额外的背景信息，留空
                    "output": "\n".join(output_content),
                    "history": [] # 基础微调通常留空，多轮对话需特殊处理
                })
        else:
            i += 1
            
    return formatted_data

# 使用示例
with open(r"D:\xwechat_files\text\texts\私聊_肖老师中秋节.json", 'r', encoding='utf-8') as f:
    raw = json.load(f)
result = convert_weflow_to_instruction(raw)
with open('train_data.jsonl', 'w', encoding='utf-8') as f:
    for item in result:
        f.write(json.dumps(item, ensure_ascii=False) + "\n")

In [2]:
from modelscope import snapshot_download

# 指定模型 ID，例如 Qwen2.5-7B-Instruct
# cache_dir 是你希望存放在本地的路径
model_dir = snapshot_download('qwen/Qwen2.5-7B-Instruct', cache_dir='D:/models/qwen')

print(f"模型已下载至: {model_dir}")

2026-04-02 14:00:46,950 - modelscope - INFO - Got 4 files, start to download ...


Processing 4 items:   0%|          | 0.00/4.00 [00:00<?, ?it/s]

2026-04-02 14:19:25,810 - modelscope - INFO - Download model 'qwen/Qwen2.5-7B-Instruct' successfully.
2026-04-02 14:19:25,811 - modelscope - INFO - Creating symbolic link [D:/models/qwen\qwen\Qwen2.5-7B-Instruct].
2026-04-02 14:19:25,814 - modelscope - WARNING - Failed to create symbolic link D:/models/qwen\qwen\Qwen2.5-7B-Instruct for D:\models\qwen\qwen\Qwen2___5-7B-Instruct.


模型已下载至: D:/models/qwen\qwen\Qwen2___5-7B-Instruct


In [ ]:
import json

# 配置参数
INPUT_FILE = "train_data.jsonl"  # 你现在的文件
OUTPUT_FILE = "final_data.json"     # 目标文件
MAX_HISTORY_LEN = 5  # 每条数据最多保留过去几轮对话作为背景

def convert_to_multiturn(input_path, output_path, max_history=5):
    combined_data = []
    current_history = []

    # 1. 读取原始单轮数据
    with open(input_path, 'r', encoding='utf-8') as f:
        lines = f.readlines()

    for line in lines:
        if not line.strip():
            continue
        
        item = json.loads(line.strip())
        
        # 2. 构造当前这一条多轮数据
        # 复制当前的 history 副本，避免引用问题
        new_entry = {
            "instruction": item["instruction"],
            "input": item.get("input", ""),
            "output": item["output"],
            "system": "你现在是 M***y，请用你一贯的语气和习惯回复朋友。", # 建议加上系统提示词
            "history": list(current_history) 
        }
        
        combined_data.append(new_entry)

        # 3. 更新 history 供给下一条使用
        # 把当前的问答对存入 history
        current_history.append([item["instruction"], item["output"]])
        
        # 4. 如果历史太长，剔除最早的一轮（滑动窗口）
        if len(current_history) > max_history:
            current_history.pop(0)

    # 5. 保存为标准的 JSON List 格式
    with open(output_path, 'w', encoding='utf-8') as f:
        json.dump(combined_data, f, ensure_ascii=False, indent=2)

    print(f"转换完成！共生成 {len(combined_data)} 条多轮数据。")

# 执行转换
convert_to_multiturn(INPUT_FILE, OUTPUT_FILE, max_history=MAX_HISTORY_LEN)

转换完成！共生成 1017 条多轮数据。


: 

## 0424没有history

### 之前的数据集有问题。我们之前没有要求上一句是“用户”，下一句是“目标人物”。这就导致连着了两句可能都是目标人物胡总和用户说的。但是我们为了学习目标人物的语气，必须要求上一句是“用户”，下一句是“目标人物”。所以我们需要重新处理数据，要求严格交替出现用户和目标人物的对话。这就会导致样本量下降

### 李旭数据集

In [7]:
import json
import random
from sklearn.model_selection import train_test_split # 需要 pip install scikit-learn

def build_dataset(data):
    session = data["session"]
    messages = data["messages"]
    nickname = session.get("nickname", "unknown")

    # 定义系统提示词和指令
    system_prompt = f"你现在是 {nickname}，请用其一贯的语气和习惯回复朋友。"
    instruction = "你是一个微信对话角色扮演模型，请严格模仿目标人物的语气、用词习惯和情绪表达方式进行回复。"

    dataset = []

    # 只保留文本消息
    clean_msgs = [
        m for m in messages
        if m.get("type") == "文本消息" and m.get("content")
    ]

    for i in range(1, len(clean_msgs)):
        current = clean_msgs[i]
        prev = clean_msgs[i - 1]

        # 过滤系统消息
        if current.get("localType") == 10000:
            continue

        # 核心逻辑：只训练目标人物（isSend=0）回复对方（isSend=1）的对话对
        # 注意：如果你的数据里 isSend 定义不同，请根据实际情况调整
        if current["isSend"] == 0 and prev["isSend"] == 1:
            sample = {
                "instruction": instruction,
                "input": prev["content"],
                "output": current["content"],
                "system": system_prompt,
                "history": [] # 如果需要多轮对话，这里需要额外的处理逻辑
            }
            dataset.append(sample)

    return dataset

def save_split_datasets(dataset, test_size=0.1, random_seed=42):
    """
    将数据集切分为训练集和测试集并保存
    """
    if not dataset:
        print("数据集为空，请检查原始 JSON 数据。")
        return

    # 使用 train_test_split 进行切分
    train_data, test_data = train_test_split(
        dataset, 
        test_size=test_size, 
        random_state=random_seed, # 固定随机种子以便复现
        shuffle=True
    )

    # 保存训练集
    with open("lixv_train_data.json", "w", encoding="utf-8") as f:
        json.dump(train_data, f, ensure_ascii=False, indent=2)
    
    # 保存测试集（用于后续评测）
    with open("lixv_test_data.json", "w", encoding="utf-8") as f:
        json.dump(test_data, f, ensure_ascii=False, indent=2)

    print(f"✅ 处理完成！")
    print(f"总样本数: {len(dataset)}")
    print(f"训练集大小: {len(train_data)}")
    print(f"测试集大小: {len(test_data)}")

# ===== 使用 =====
if __name__ == "__main__":
    # 读取原始数据
    file_path = r"D:\xwechat_files\text\texts\私聊_李旭（二月初九）.json"
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    # 构建样本
    full_dataset = build_dataset(raw_data)

    # 执行切分并保存（10% 作为测试集）
    save_split_datasets(full_dataset, test_size=0.1)

✅ 处理完成！
总样本数: 2904
训练集大小: 2613
测试集大小: 291


### 肖老师数据集

In [6]:
import json
import random
from sklearn.model_selection import train_test_split # 需要 pip install scikit-learn

def build_dataset(data):
    session = data["session"]
    messages = data["messages"]
    nickname = session.get("nickname", "unknown")

    # 定义系统提示词和指令
    system_prompt = f"你现在是 {nickname}，请用其一贯的语气和习惯回复朋友。"
    instruction = "你是一个微信对话角色扮演模型，请严格模仿目标人物的语气、用词习惯和情绪表达方式进行回复。"

    dataset = []

    # 只保留文本消息
    clean_msgs = [
        m for m in messages
        if m.get("type") == "文本消息" and m.get("content")
    ]

    for i in range(1, len(clean_msgs)):
        current = clean_msgs[i]
        prev = clean_msgs[i - 1]

        # 过滤系统消息
        if current.get("localType") == 10000:
            continue

        # 核心逻辑：只训练目标人物（isSend=0）回复对方（isSend=1）的对话对
        # 注意：如果你的数据里 isSend 定义不同，请根据实际情况调整
        if current["isSend"] == 0 and prev["isSend"] == 1:
            sample = {
                "instruction": instruction,
                "input": prev["content"],
                "output": current["content"],
                "system": system_prompt,
                "history": [] # 如果需要多轮对话，这里需要额外的处理逻辑
            }
            dataset.append(sample)

    return dataset

def save_split_datasets(dataset, test_size=0.1, random_seed=42):
    """
    将数据集切分为训练集和测试集并保存
    """
    if not dataset:
        print("数据集为空，请检查原始 JSON 数据。")
        return

    # 使用 train_test_split 进行切分
    train_data, test_data = train_test_split(
        dataset, 
        test_size=test_size, 
        random_state=random_seed, # 固定随机种子以便复现
        shuffle=True
    )

    # 保存训练集
    with open("xiaolaoshi_train_data.json", "w", encoding="utf-8") as f:
        json.dump(train_data, f, ensure_ascii=False, indent=2)
    
    # 保存测试集（用于后续评测）
    with open("xiaolaoshi_test_data.json", "w", encoding="utf-8") as f:
        json.dump(test_data, f, ensure_ascii=False, indent=2)

    print(f"✅ 处理完成！")
    print(f"总样本数: {len(dataset)}")
    print(f"训练集大小: {len(train_data)}")
    print(f"测试集大小: {len(test_data)}")

# ===== 使用 =====
if __name__ == "__main__":
    # 读取原始数据
    file_path = r"D:\xwechat_files\text\texts\私聊_肖老师中秋节.json"
    with open(file_path, "r", encoding="utf-8") as f:
        raw_data = json.load(f)

    # 构建样本
    full_dataset = build_dataset(raw_data)

    # 执行切分并保存（10% 作为测试集）
    save_split_datasets(full_dataset, test_size=0.1)

✅ 处理完成！
总样本数: 1047
训练集大小: 942
测试集大小: 105
